In [21]:
import os
import sys
os.chdir("..")

import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_DB_PATH, NSE_DB_PATH

In [22]:
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

# read from nse, write to research.db
con.execute("""
    CREATE OR REPLACE TABLE forward_returns AS
    WITH daily AS (
        SELECT
            trade_date,
            index_name,
            close
        FROM nse.market_activity_index
        WHERE index_name = 'Nifty 50'   -- adjust to exact name in your data
        ORDER BY trade_date
    )
    SELECT
        d.trade_date,
        d.index_name,
        d.close,

        -- forward returns
        ROUND((f1.close - d.close) / d.close * 100, 4) AS fwd_ret_1d,
        ROUND((f5.close - d.close) / d.close * 100, 4) AS fwd_ret_5d,
        ROUND((f20.close - d.close) / d.close * 100, 4) AS fwd_ret_20d,

        -- direction label (for classification)
        CASE WHEN f1.close > d.close THEN 1 ELSE 0 END AS up_1d

    FROM daily d
    LEFT JOIN daily f1
        ON f1.trade_date = (
            SELECT MIN(trade_date) FROM daily
            WHERE trade_date > d.trade_date
        )
    LEFT JOIN daily f5
        ON f5.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 4
        )
    LEFT JOIN daily f20
        ON f20.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 19
        )
    ;
""")

In [23]:
results = con.execute(""" SELECT * FROM forward_returns ORDER BY trade_date DESC LIMIT 10 OFFSET 20; """).fetchall()
for row in results:
    print(row)

(datetime.date(2026, 5, 21), 'Nifty 50', 23654.7, 0.2731, -0.4521, 1.5151, 1)
(datetime.date(2026, 5, 20), 'Nifty 50', 23659.0, -0.0182, 1.0489, 2.1514, 0)
(datetime.date(2026, 5, 19), 'Nifty 50', 23618.0, 0.1736, 1.252, 1.9803, 1)
(datetime.date(2026, 5, 18), 'Nifty 50', 23649.95, -0.1351, 1.6142, 1.4343, 0)
(datetime.date(2026, 5, 15), 'Nifty 50', 23643.5, 0.0273, 0.3206, 0.8899, 1)
(datetime.date(2026, 5, 14), 'Nifty 50', 23689.6, -0.1946, -0.1473, -0.2816, 0)
(datetime.date(2026, 5, 13), 'Nifty 50', 23412.6, 1.1831, 1.0524, -1.0721, 1)
(datetime.date(2026, 5, 12), 'Nifty 50', 23379.55, 0.1414, 1.0199, -0.704, 1)
(datetime.date(2026, 5, 11), 'Nifty 50', 23815.85, -1.832, -0.6966, -2.4091, 0)
(datetime.date(2026, 5, 8), 'Nifty 50', 24176.15, -1.4903, -2.2032, -4.3562, 0)


In [24]:
results = con.execute(""" SELECT DISTINCT index_name FROM nse.market_activity_index ORDER BY 1; """).fetchall()
for row in results:
    print(row)

('BHARATBOND-APR25',)
('BHARATBOND-APR30',)
('BHARATBOND-APR31',)
('BHARATBOND-APR32',)
('BHARATBOND-APR33',)
('India VIX',)
('NIFTY Alpha 50',)
('NIFTY AlphaLowVol',)
('NIFTY CONSR DURBL',)
('NIFTY HEALTHCARE',)
('NIFTY IND DIGITAL',)
('NIFTY INDIA MFG',)
('NIFTY LARGEMID250',)
('NIFTY M150 QLTY50',)
('NIFTY MICROCAP250',)
('NIFTY MID SELECT',)
('NIFTY MIDCAP 100',)
('NIFTY MIDCAP 150',)
('NIFTY MIDSML 400',)
('NIFTY OIL AND GAS',)
('NIFTY SMLCAP 100',)
('NIFTY SMLCAP 250',)
('NIFTY SMLCAP 50',)
('NIFTY TOTAL MKT',)
('NIFTY100 EQL Wgt',)
('NIFTY100 ESG',)
('NIFTY100 LowVol30',)
('NIFTY100 Qualty30',)
('NIFTY200 QUALTY30',)
('NIFTY50 EQL Wgt',)
('NIFTY500 MULTICAP',)
('Nifty 100',)
('Nifty 200',)
('Nifty 50',)
('Nifty 500',)
('Nifty AQL 30',)
('Nifty AQLV 30',)
('Nifty Auto',)
('Nifty Bank',)
('Nifty CPSE',)
('Nifty Capital Mkt',)
('Nifty Cement',)
('Nifty Chemicals',)
('Nifty Commodities',)
('Nifty Consumption',)
('Nifty CoreHousing',)
('Nifty Corp MAATR',)
('Nifty Div Opps 50',)
('Ni

In [25]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close         AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close        AS vix_close

FROM forward_returns fr
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

ORDER BY fr.trade_date
""")

In [26]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10 OFFSET 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 5, 21), 23654.7, 0.2731, -0.4521, 1.5151, 1, 17.8225)
(datetime.date(2026, 5, 20), 23659.0, -0.0182, 1.0489, 2.1514, 0, 18.44)
(datetime.date(2026, 5, 19), 23618.0, 0.1736, 1.252, 1.9803, 1, 18.675)
(datetime.date(2026, 5, 18), 23649.95, -0.1351, 1.6142, 1.4343, 0, 19.63)
(datetime.date(2026, 5, 15), 23643.5, 0.0273, 0.3206, 0.8899, 1, 18.79)
(datetime.date(2026, 5, 14), 23689.6, -0.1946, -0.1473, -0.2816, 0, 18.6125)
(datetime.date(2026, 5, 13), 23412.6, 1.1831, 1.0524, -1.0721, 1, 19.425)
(datetime.date(2026, 5, 12), 23379.55, 0.1414, 1.0199, -0.704, 1, 19.28)
(datetime.date(2026, 5, 11), 23815.85, -1.832, -0.6966, -2.4091, 0, 18.5525)
(datetime.date(2026, 5, 8), 24176.15, -1.4903, -2.2032, -4.3562, 0, 16.84)


In [27]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 13 THEN '1_low <13'
            WHEN vix_close < 16 THEN '2_calm 13-16'
            WHEN vix_close < 20 THEN '3_normal 16-20'
            WHEN vix_close < 25 THEN '4_elevated 20-25'
            ELSE                     '5_fear >25'
        END AS vix_regime,

        COUNT(*)                        AS days,
        ROUND(AVG(fwd_ret_1d), 3)       AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)       AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)      AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)      AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL

    GROUP BY 1
    ORDER BY 1
""").fetchall()

for row in results: print(row)

('1_low <13', 167, -0.023, -0.165, -0.533, 50.3)
('2_calm 13-16', 188, -0.042, -0.065, -0.377, 47.3)
('3_normal 16-20', 85, -0.075, -0.114, -0.09, 49.4)
('4_elevated 20-25', 20, 0.635, 1.275, 3.694, 65.0)
('5_fear >25', 6, 0.524, 4.01, 6.276, 83.3)


In [28]:
results = con.execute("""
    SELECT *
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchall()
for row in results: print(row)

('IDO', 'NIFTY', datetime.date(2026, 6, 23), datetime.date(2026, 6, 19), 120265925.0, 154919700.0, 0.7763113729241665, 24013.1, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 6, 30), datetime.date(2026, 6, 19), 82423145.0, 80713185.0, 1.0211856340447971, 24013.09999999989, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 7), datetime.date(2026, 6, 19), 3038165.0, 3521765.0, 0.8626824901718314, 24013.10000000001, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 14), datetime.date(2026, 6, 19), 625300.0, 697515.0, 0.8964681763116206, 24013.100000000013, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 21), datetime.date(2026, 6, 19), 50830.0, 61555.0, 0.8257655755015839, 24013.10000000006, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 28), datetime.date(2026, 6, 19), 18974735.0, 16020095.0, 1.1844333632228772, 24013.09999999999, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 25), datetime.date(2026, 6, 19), 4140695.0, 3074110.0, 1.3469573307396288, 24013.1, 24000.0)
('IDO', 'NIFTY', 

In [29]:
results = con.execute("""
    SELECT
        expiry,
        trade_date,
        pe_oi,
        ce_oi,
        pe_oi + ce_oi AS total_oi
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND trade_date = '2026-06-10'
    ORDER BY total_oi DESC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 16), datetime.date(2026, 6, 10), 93749955.0, 114460970.0, 208210925.0)
(datetime.date(2026, 6, 30), datetime.date(2026, 6, 10), 69249000.0, 66301370.0, 135550370.0)
(datetime.date(2026, 12, 29), datetime.date(2026, 6, 10), 11812480.0, 9371595.0, 21184075.0)
(datetime.date(2026, 6, 23), datetime.date(2026, 6, 10), 9232405.0, 10127780.0, 19360185.0)
(datetime.date(2026, 7, 28), datetime.date(2026, 6, 10), 9825335.0, 8697390.0, 18522725.0)
(datetime.date(2026, 9, 29), datetime.date(2026, 6, 10), 5249010.0, 4500755.0, 9749765.0)
(datetime.date(2026, 8, 25), datetime.date(2026, 6, 10), 2054260.0, 1737060.0, 3791320.0)
(datetime.date(2026, 7, 7), datetime.date(2026, 6, 10), 275860.0, 505245.0, 781105.0)
(datetime.date(2027, 12, 28), datetime.date(2026, 6, 10), 218430.0, 155535.0, 373965.0)
(datetime.date(2028, 12, 26), datetime.date(2026, 6, 10), 44980.0, 16115.0, 61095.0)
(datetime.date(2026, 7, 14), datetime.date(2026, 6, 10), 9425.0, 12155.0, 21580.0)
(datetime.dat

In [30]:
results = con.execute("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY total_oi) AS p25,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY total_oi) AS p50,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY total_oi) AS p75,
        percentile_cont(0.90) WITHIN GROUP (ORDER BY total_oi) AS p90,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY total_oi) AS p95,
        MIN(total_oi)  AS min_oi,
        MAX(total_oi)  AS max_oi,
        COUNT(*)       AS total_rows
    FROM (
        SELECT pe_oi + ce_oi AS total_oi
        FROM nse.options_analytics
        WHERE ticker = 'NIFTY'
          AND pe_oi IS NOT NULL
          AND ce_oi IS NOT NULL
    )
""").fetchall()
for row in results: print(row)

(2825.0, 323485.0, 11805262.5, 86638417.5, 210763475.0, 0.0, 424741950.0, 8406)


In [31]:
results = con.execute("""
    SELECT
        trade_date,
        COUNT(*) AS liquid_expiries
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 19), 4)
(datetime.date(2026, 6, 18), 4)
(datetime.date(2026, 6, 17), 4)
(datetime.date(2026, 6, 16), 5)
(datetime.date(2026, 6, 15), 5)
(datetime.date(2026, 6, 12), 5)
(datetime.date(2026, 6, 11), 5)
(datetime.date(2026, 6, 10), 5)
(datetime.date(2026, 6, 9), 5)
(datetime.date(2026, 6, 8), 5)
(datetime.date(2026, 6, 5), 5)
(datetime.date(2026, 6, 4), 5)
(datetime.date(2026, 6, 3), 5)
(datetime.date(2026, 6, 2), 5)
(datetime.date(2026, 6, 1), 5)
(datetime.date(2026, 5, 29), 4)
(datetime.date(2026, 5, 27), 4)
(datetime.date(2026, 5, 26), 5)
(datetime.date(2026, 5, 25), 4)
(datetime.date(2026, 5, 22), 4)


In [32]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close              AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close             AS vix_close,
    pcr_agg.pcr,
    mp_agg.max_pain_dist_pct

FROM forward_returns fr

LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- PCR: all expiries
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(
            SUM(
                ((underlying - max_pain) / NULLIF(max_pain, 0) * 100)
                * (pe_oi + ce_oi)
            ) / NULLIF(SUM(pe_oi + ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [33]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 19), 24013.1, None, None, None, 0, 12.97, 0.91, -0.1318)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, None, None, 0, 12.6725, 1.1229, 0.2006)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, None, None, 1, 13.1875, 1.1, 0.1268)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, None, None, 1, 13.3625, 1.1117, 0.0237)
(datetime.date(2026, 6, 15), 23853.9, 0.567, None, None, 1, 14.3525, 0.9881, -0.1728)
(datetime.date(2026, 6, 12), 23622.9, 0.9779, 1.6518, None, 1, 14.7175, 1.412, -0.3175)
(datetime.date(2026, 6, 11), 23161.6, 1.9917, 4.3451, None, 1, 15.6125, 0.9853, -1.6451)
(datetime.date(2026, 6, 10), 23214.95, -0.2298, 3.7508, None, 0, 15.6325, 0.9344, -1.8159)
(datetime.date(2026, 6, 9), 23242.1, -0.1168, 3.2142, None, 0, 15.575, 0.9304, -0.9622)
(datetime.date(2026, 6, 8), 23123.0, 0.5151, 3.1609, None, 1, 17.0275, 0.7764, -1.5009)


In [34]:
results = con.execute("""
    SELECT
        CASE
            WHEN pcr < 0.7  THEN '1_very_low <0.7'
            WHEN pcr < 0.9  THEN '2_low 0.7-0.9'
            WHEN pcr < 1.1  THEN '3_neutral 0.9-1.1'
            WHEN pcr < 1.3  THEN '4_high 1.1-1.3'
            ELSE                 '5_very_high >1.3'
        END AS pcr_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND pcr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== PCR ===")
for row in results: print(row)

results = con.execute("""
    SELECT
        CASE
            WHEN max_pain_dist_pct < -3   THEN '1_far_below <-3%'
            WHEN max_pain_dist_pct < -1.5 THEN '2_below -3 to -1.5%'
            WHEN max_pain_dist_pct < 0    THEN '3_slightly_below -1.5 to 0%'
            WHEN max_pain_dist_pct < 1.5  THEN '4_slightly_above 0 to 1.5%'
            ELSE                               '5_far_above >1.5%'
        END AS mp_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND max_pain_dist_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Max Pain Distance ===")
for row in results: print(row)

=== PCR ===
('1_very_low <0.7', 25, 0.025, 0.112, -1.021, 52.0)
('2_low 0.7-0.9', 175, -0.072, -0.026, 0.189, 42.9)
('3_neutral 0.9-1.1', 155, 0.021, -0.007, -0.099, 51.0)
('4_high 1.1-1.3', 92, 0.045, 0.112, -0.191, 56.5)
('5_very_high >1.3', 19, 0.112, -0.347, -1.541, 73.7)

=== Max Pain Distance ===
('1_far_below <-3%', 5, 0.292, 0.385, 5.101, 80.0)
('2_below -3 to -1.5%', 28, 0.099, 0.58, 1.701, 67.9)
('3_slightly_below -1.5 to 0%', 288, -0.001, 0.068, -0.209, 46.5)
('4_slightly_above 0 to 1.5%', 144, -0.035, -0.263, -0.399, 52.8)
('5_far_above >1.5%', 1, -1.39, 0.083, 0.716, 0.0)


In [35]:
# Futures - what tickers and how many rows
results = con.execute("""
    SELECT instrument_type, ticker, COUNT(*) as rows, 
           MIN(trade_date) as from_date, MAX(trade_date) as to_date
    FROM nse.futures_analytics
    GROUP BY instrument_type, ticker
    ORDER BY rows DESC
    LIMIT 10
""").fetchall()
print("=== Futures ===")
for row in results: print(row)

=== Futures ===
('STF', 'SUNPHARMA', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'BEL', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'NTPC', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'GODREJCP', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'ALKEM', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'HAVELLS', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('IDF', 'NIFTYNXT50', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'TVSMOTOR', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'CUMMINSIND', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))
('STF', 'DALBHARAT', 1401, datetime.date(2024, 8, 1), datetime.date(2026, 6, 19))


In [36]:
# Participant - what participant types and asset classes exist
results = con.execute("""
    SELECT participant_type, metric_type, asset_class, direction, option_side,
           COUNT(*) as rows
    FROM nse.participant_activity
    GROUP BY participant_type, metric_type, asset_class, direction, option_side
    ORDER BY participant_type, metric_type, asset_class
    LIMIT 30
""").fetchall()
print("\n=== Participant ===")
for row in results: print(row)


=== Participant ===
('Client', 'OI', 'INDEX', 'long', 'NA', 467)
('Client', 'OI', 'INDEX', 'long', 'PE', 467)
('Client', 'OI', 'INDEX', 'short', 'PE', 467)
('Client', 'OI', 'INDEX', 'short', 'CE', 467)
('Client', 'OI', 'INDEX', 'short', 'NA', 467)
('Client', 'OI', 'INDEX', 'long', 'CE', 467)
('Client', 'OI', 'STOCK', 'long', 'CE', 467)
('Client', 'OI', 'STOCK', 'short', 'PE', 467)
('Client', 'OI', 'STOCK', 'short', 'CE', 467)
('Client', 'OI', 'STOCK', 'short', 'NA', 467)
('Client', 'OI', 'STOCK', 'long', 'PE', 467)
('Client', 'OI', 'STOCK', 'long', 'NA', 467)
('Client', 'VOL', 'INDEX', 'long', 'NA', 467)
('Client', 'VOL', 'INDEX', 'short', 'NA', 467)
('Client', 'VOL', 'INDEX', 'long', 'PE', 467)
('Client', 'VOL', 'INDEX', 'short', 'CE', 467)
('Client', 'VOL', 'INDEX', 'long', 'CE', 467)
('Client', 'VOL', 'INDEX', 'short', 'PE', 467)
('Client', 'VOL', 'STOCK', 'long', 'CE', 467)
('Client', 'VOL', 'STOCK', 'long', 'NA', 467)
('Client', 'VOL', 'STOCK', 'short', 'NA', 467)
('Client', 'VOL

In [38]:
results = con.execute("""
    SELECT trade_date, expiry, basis, cost_of_carry, 
           chng_oi_per, open_int
    FROM nse.futures_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDF'
      AND trade_date = '2026-06-10'
    ORDER BY expiry ASC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 10), datetime.date(2026, 6, 30), 25.149999999997817, 0.019771203470175906, -1.4561936771066268, 19134310.0)
(datetime.date(2026, 6, 10), datetime.date(2026, 7, 28), 129.95000000000073, 0.042565737093267005, 0.20905923344947736, 1682460.0)
(datetime.date(2026, 6, 10), datetime.date(2026, 8, 25), 232.45000000000073, 0.04808848222918073, 1.3488880787458988, 542100.0)


In [39]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close             AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,

    -- VIX
    vix.close            AS vix_close,

    -- PCR (all expiries)
    pcr_agg.pcr,

    -- Max pain distance (liquid expiries, OI-weighted)
    mp_agg.max_pain_dist_pct,

    -- Futures near month
    fut.basis,
    fut.cost_of_carry,
    fut.chng_oi_per      AS fut_chng_oi_pct,

    -- FII net futures positioning (long - short, futures only)
    ROUND(
        (fii_long_fut.contracts - fii_short_fut.contracts) /
        NULLIF(fii_long_fut.contracts + fii_short_fut.contracts, 0) * 100
    , 2)                 AS fii_fut_net_pct,

    -- Client net futures positioning (retail, often contrarian)
    ROUND(
        (cli_long_fut.contracts - cli_short_fut.contracts) /
        NULLIF(cli_long_fut.contracts + cli_short_fut.contracts, 0) * 100
    , 2)                 AS client_fut_net_pct,

    -- FII put buying (protection signal)
    ROUND(
        (fii_long_pe.contracts - fii_short_pe.contracts) /
        NULLIF(fii_long_pe.contracts + fii_short_pe.contracts, 0) * 100
    , 2)                 AS fii_pe_net_pct

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- PCR: all expiries
LEFT JOIN (
    SELECT trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT trade_date,
        ROUND(
            SUM(((underlying - max_pain) / NULLIF(max_pain, 0) * 100) * (pe_oi + ce_oi))
            / NULLIF(SUM(pe_oi + ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

-- Futures: near month only
LEFT JOIN (
    SELECT trade_date, basis, cost_of_carry, chng_oi_per
    FROM nse.futures_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
      AND (trade_date, expiry) IN (
          SELECT trade_date, MIN(expiry)
          FROM nse.futures_analytics
          WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
          GROUP BY trade_date
      )
) fut ON fut.trade_date = fr.trade_date

-- Participant: FII long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) fii_long_fut ON fii_long_fut.trade_date = fr.trade_date

-- Participant: FII short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) fii_short_fut ON fii_short_fut.trade_date = fr.trade_date

-- Participant: Client long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) cli_long_fut ON cli_long_fut.trade_date = fr.trade_date

-- Participant: Client short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) cli_short_fut ON cli_short_fut.trade_date = fr.trade_date

-- Participant: FII long PE (put buying = hedging/bearish)
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'PE'
) fii_long_pe ON fii_long_pe.trade_date = fr.trade_date

-- Participant: FII short PE
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'PE'
) fii_short_pe ON fii_short_pe.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [40]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 5
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 19), 24013.1, None, None, None, 0, 12.97, 0.91, -0.1318, 43.80000000000291, 0.060523782283992196, -0.16770465085151448, -74.09, 51.6, 37.34)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, None, None, 0, 12.6725, 1.1229, 0.2006, 24.5, 0.030834505682445106, -1.7608316477872616, -73.05, 51.29, 35.21)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, None, None, 1, 13.1875, 1.1, 0.1268, 8.299999999999272, 0.009675386704079228, -2.1750151331719128, -73.82, 50.99, 36.74)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, None, None, 1, 13.3625, 1.1117, 0.0237, 11.849999999998545, 0.012878590053061098, -3.575780127096694, -74.58, 51.43, 39.58)
(datetime.date(2026, 6, 15), 23853.9, 0.567, None, None, 1, 14.3525, 0.9881, -0.1728, 62.69999999999709, 0.06396019099601864, -2.099285714285714, -74.6, 52.38, 31.24)


In [41]:
queries = {
    "Basis": ("basis", [
        ("1_discount <0",      "basis < 0"),
        ("2_low 0-50",         "basis >= 0 AND basis < 50"),
        ("3_mid 50-100",       "basis >= 50 AND basis < 100"),
        ("4_high >100",        "basis >= 100"),
    ]),
    "FII Fut Net": ("fii_fut_net_pct", [
        ("1_very_short <-50",  "fii_fut_net_pct < -50"),
        ("2_short -50 to -20", "fii_fut_net_pct >= -50 AND fii_fut_net_pct < -20"),
        ("3_neutral -20 to 20","fii_fut_net_pct >= -20 AND fii_fut_net_pct < 20"),
        ("4_long 20 to 50",    "fii_fut_net_pct >= 20 AND fii_fut_net_pct < 50"),
        ("5_very_long >50",    "fii_fut_net_pct >= 50"),
    ]),
    "Client Fut Net": ("client_fut_net_pct", [
        ("1_very_short <-50",  "client_fut_net_pct < -50"),
        ("2_short -50 to -20", "client_fut_net_pct >= -50 AND client_fut_net_pct < -20"),
        ("3_neutral -20 to 20","client_fut_net_pct >= -20 AND client_fut_net_pct < 20"),
        ("4_long 20 to 50",    "client_fut_net_pct >= 20 AND client_fut_net_pct < 50"),
        ("5_very_long >50",    "client_fut_net_pct >= 50"),
    ]),
    "FII PE Net": ("fii_pe_net_pct", [
        ("1_very_short <-50",  "fii_pe_net_pct < -50"),
        ("2_short -50 to -20", "fii_pe_net_pct >= -50 AND fii_pe_net_pct < -20"),
        ("3_neutral -20 to 20","fii_pe_net_pct >= -20 AND fii_pe_net_pct < 20"),
        ("4_long 20 to 50",    "fii_pe_net_pct >= 20 AND fii_pe_net_pct < 50"),
        ("5_very_long >50",    "fii_pe_net_pct >= 50"),
    ]),
}

for factor_name, (col, buckets) in queries.items():
    case_sql = "CASE\n" + "\n".join(
        f"  WHEN {cond} THEN '{label}'" for label, cond in buckets
    ) + "\n  ELSE 'other' END"

    results = con.execute(f"""
        SELECT
            {case_sql} AS bucket,
            COUNT(*)                    AS days,
            ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
            ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
            ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
            ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND {col} IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """).fetchall()

    print(f"\n=== {factor_name} ===")
    for row in results: print(row)


=== Basis ===
('1_discount <0', 39, 0.061, 0.459, -0.139, 64.1)
('2_low 0-50', 152, 0.1, 0.159, 0.206, 53.3)
('3_mid 50-100', 182, -0.087, -0.089, -0.225, 45.6)
('4_high >100', 93, -0.044, -0.269, -0.403, 47.3)

=== FII Fut Net ===
('1_very_short <-50', 321, -0.011, 0.028, -0.041, 49.2)
('2_short -50 to -20', 76, 0.044, -0.261, -0.414, 52.6)
('3_neutral -20 to 20', 40, 0.048, 0.484, 0.765, 50.0)
('4_long 20 to 50', 22, -0.084, 0.46, 0.172, 54.5)
('5_very_long >50', 7, -0.332, -2.572, -5.912, 42.9)

=== Client Fut Net ===
('2_short -50 to -20', 12, -0.037, -1.034, -4.158, 58.3)
('3_neutral -20 to 20', 97, 0.035, 0.38, 1.304, 52.6)
('4_long 20 to 50', 331, -0.025, -0.175, -0.486, 48.0)
('5_very_long >50', 26, 0.112, 1.55, 2.355, 61.5)

=== FII PE Net ===
('3_neutral -20 to 20', 185, -0.051, -0.226, -0.88, 48.6)
('4_long 20 to 50', 280, 0.027, 0.15, 0.423, 51.1)
('5_very_long >50', 1, -0.302, 0.953, 1.592, 0.0)


In [42]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_discount', 11, -0.486, 1.073, 1.22, 54.5)
('VIX_high', 'basis_high', 6, 0.082, 1.017, 0.964, 66.7)
('VIX_high', 'basis_low', 25, 0.375, 0.278, 1.399, 56.0)
('VIX_high', 'basis_mid', 22, 0.226, 1.195, 2.519, 59.1)
('VIX_low', 'basis_discount', 11, 0.235, 0.243, -1.734, 72.7)
('VIX_low', 'basis_high', 61, -0.095, -0.438, -0.358, 44.3)
('VIX_low', 'basis_low', 77, -0.005, -0.06, -0.443, 53.2)
('VIX_low', 'basis_mid', 95, -0.078, -0.077, -0.501, 49.5)
('VIX_mid', 'basis_discount', 17, 0.302, 0.201, 0.022, 64.7)
('VIX_mid', 'basis_high', 26, 0.048, -0.169, -0.879, 50.0)
('VIX_mid', 'basis_low', 50, 0.124, 0.424, 0.602, 52.0)
('VIX_mid', 'basis_mid', 65, -0.206, -0.547, -0.824, 35.4)


In [43]:
con.close()